# Import and Functions

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from math import pi
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, cohen_kappa_score, matthews_corrcoef)

In [ ]:
with open("final_all_models_outs.pkl", "rb") as f:
    model_dfs = pickle.load(f)

df_base = pd.read_csv("final_total_dataset.csv")

df_aya_persian = pd.read_csv("aya_persian.csv")

In [ ]:
df_base = df_base.drop(columns=["medical_history"])
df_base = df_base.drop_duplicates(subset=['id'] , keep='first')

In [ ]:
model_dfs = {
    k.replace("aya_", "").replace("-00000-of-00001", ""):v
    for k, v in model_dfs.items()
}
model_dfs['expanse_farsi'] = df_aya_persian


for k , df in model_dfs.items():
    model_dfs[k] = df.drop_duplicates(subset=['id'] , keep='first').sort_values('id')

In [ ]:
new_order = ['Llama_large', 'qwen_7b', 'Llama_mini', 'qwen1_5', 'gemma_1b', 'expanse','expanse_farsi']
model_dfs = {key: model_dfs[key] for key in new_order if key in model_dfs}

In [ ]:
model_name_mapper = {
    "qwen_7b":"Qwen2.5-7b-Instruct" ,
    'Llama_large': "Llama-3.1-8B-Instruct",
    'Llama_mini' : "Llama-3.2-3B-Instruct",
    'gemma_1b': "Gemma-3-1b-it",
    'qwen1_5' : "Qwen2.5-1.5B-Instruct",
    'expanse': "Aya-expanse-8b (English)" ,
    'expanse_farsi': "Aya-expanse-8b (Persian)" ,
}

colors_base = {
    "Llama-3.1-8B-Instruct": "#4DD42B",
    "Qwen2.5-7b-Instruct": "#F11528",
    "Llama-3.2-3B-Instruct": '#8338EC',
    "Qwen2.5-1.5B-Instruct": "#CA366860",
    "Gemma-3-1b-it": '#F77F00',
    "Aya-expanse-8b (English)": '#06AED5',
    "Aya-expanse-8b (Persian)": '#264653',
}

In [ ]:
model_dfs = {model_name_mapper.get(k, k): v for k, v in model_dfs.items()}

In [ ]:
def calculate_metrics_for_feature(df1_col, df_base_col):
    """
    Calculate all metrics for a single feature, handling missing values.
    """
    
    valid_mask = df1_col.notna()
    missing_count = (~valid_mask).sum()

    df1_col = df1_col.fillna(False)
    y_true = df_base_col.values.astype(bool)
    y_pred = df1_col.values.astype(bool)

    if len(y_true) == 0:
        return {
            'Accuracy': np.nan,
            'Precision': np.nan, 
            'Recall': np.nan,    
            'Sensitivity': np.nan,
            'Specificity': np.nan,
            'F1-score': np.nan,
            'MCC': np.nan,
            'Missing_count': len(df1_col)
        }

    # Accuracy
    accuracy = accuracy_score(y_true, y_pred) 

    # Precision
    precision = precision_score(y_true, y_pred, zero_division=0) 

    # Recall (True Positive Rate/Sensitivity)
    sensitivity = recall_score(y_true, y_pred, zero_division=0)


    # Specificity (True Negative Rate)
    tn = ((y_true == False) & (y_pred == False)).sum()
    fp = ((y_true == False) & (y_pred == True)).sum()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

    # F1-score
    f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)

    kappa = cohen_kappa_score(y_true, y_pred)

    mcc = matthews_corrcoef(y_true, y_pred)

    return {
        'Accuracy': accuracy,      
        'Precision': precision,     
        'Sensitivity': sensitivity, 
        'Specificity': specificity,
        'F1-score': f1,
        'MCC': mcc,
        'Missing_count': missing_count
    }

def get_analysis(df1, df_base):
    results = {}

    for col in df1.columns:
        results[col] = calculate_metrics_for_feature(df1[col], df_base[col])

    results_df = pd.DataFrame(results).T

    results_df.columns = ['Accuracy', 'Precision',  'Sensitivity', 'Specificity', 'F1-score', 'MCC', 'Missing_count']

    results_df = results_df.reset_index()
    results_df.rename(columns={'index': 'Feature'}, inplace=True)

    numeric_cols = ['Accuracy', 'Precision', 'Sensitivity', 'Specificity', 'F1-score', 'MCC', 'Missing_count']
    results_df[numeric_cols] = results_df[numeric_cols].round(4)

    return results_df

In [ ]:
def get_model_analysis(model_dfs , df_base):
    df_base = df_base.drop(columns=["id"])
    final_analysis = {}
    for k , df in model_dfs.items():
        df1 = df.drop_duplicates(subset=['id'] , keep='first').reset_index(drop = True).sort_values('id').drop(columns=["id"]).reset_index(drop=True)
        df_base = df_base.reset_index(drop=True)
        df_analysis = get_analysis(df1 , df_base)
        final_analysis[k] = df_analysis
    return final_analysis

def get_metrics(final_analysis): 
    result_dict = {}
    metrics = [ 'Accuracy' ,'Precision',  'Sensitivity', 'Specificity',  'F1-score', 'MCC', 'Missing_count']

    first_key = list(final_analysis.keys())[0]
    features = final_analysis[first_key]['Feature'].tolist()

    for metric in metrics:
        metric_df = pd.DataFrame(index=features, columns=final_analysis.keys())
        for model_name, df in final_analysis.items():
            df_indexed = df.set_index('Feature')
            metric_df[model_name] = df_indexed[metric]

        result_dict[metric] = metric_df.reset_index().rename(columns={'index': 'features'})
    return result_dict

In [ ]:
df_base.columns

In [ ]:
final_analysis = get_model_analysis(model_dfs , df_base)

In [ ]:
result_dict = get_metrics(final_analysis)

# Tables

In [ ]:
reverse_mapping = {
    "com_request_for_visit": "Doctor's visit request",
    "com_psychiatric_psychological_complaints": "Psychological complaints",
    "com_sleep_disorders": "Sleep disorders",
    "com_loss_of_appetite": "Loss of appetite",
    "com_seizures": "Seizures",
    "com_weakness_and_fatigue": "Weakness and fatigue",
    "com_decreased_level_of_consciousness": "Decreased level of consciousness",
    "com_fever": "Fever",
    "com_shortness_of_breath_oxygen_saturation_drop_respiratory_complaints": "Respiratory complaints",
    "com_issues_related_to_insurance_and_treatment_costs": "Insurance/treatment cost issues",
    "com_urinary_issues": "Urinary tract issues",
    "com_pain": "Pain",
    "com_gastrointestinal_issues": "Gastrointestinal issues"
}

In [ ]:
for metric in result_dict:
    if "features" in result_dict[metric]:
        result_dict[metric]["features"] = result_dict[metric]["features"].replace(reverse_mapping)


In [ ]:
# import pandas as pd

# with pd.ExcelWriter(r"C:\Users\Fiasco\Desktop\ResBox_new\docs\metrics.xlsx") as writer:
#     for k, df in result_dict.items():
#         df.to_excel(writer, sheet_name=k, index=False)


# Figures

In [ ]:
sns.set_theme(style="whitegrid", context="talk") # 
plt.rcParams['font.family'] = 'Arial'

## Spider

In [ ]:
def get_spider(final_analysis , colors = colors_base):

    metrics = ['Sensitivity','Specificity', 'F1-score','Precision','Accuracy'  ]

    model_means = {}
    for model_name, df in final_analysis.items():
        metrics = metrics
        means = df[metrics].median()
        model_means[model_name] = means

    means_df = pd.DataFrame(model_means).T

    metrics_to_plot = metrics
    plot_df = means_df[metrics_to_plot].copy()


    categories = metrics
    N = len(categories)

    angles = [n / float(N) * 2 * pi for n in range(N)]
    angles += angles[:1]

    fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(projection='polar'), dpi=330)


    # Plot each model
    for idx, (model_name, row) in enumerate(plot_df.iterrows()):
        values = row.tolist()
        values += values[:1]  
        
        color = colors.get(model_name, '#808080')

        ax.plot(angles, values, 'o-', linewidth=2.5, label=model_name,
                color=color, markersize=6, markeredgewidth=1.5,
                markeredgecolor='white', alpha=0.85)
        ax.fill(angles, values, alpha=0.08, color=color)


    ax.set_xticks(angles[:-1])
    ax.set_xticklabels([]) 

    ax.set_ylim(0, 1)
    ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
    ax.set_yticklabels(['0.2', '0.4', '0.6', '0.8', '1.0'], size=10, color='gray')

    ax.set_theta_offset(pi / 2)  # Start from top
    ax.set_theta_direction(-1)  # Clockwise

    ax.grid(True, linestyle='-', alpha=0.3, linewidth=0.8, color='gray')
    fig.patch.set_alpha(0.0)
    ax.patch.set_alpha(0.0)

    for angle in angles[:-1]:
        ax.plot([angle, angle], [0, 1], color='gray', linewidth=0.5, alpha=0.2)

    plt.legend(fontsize=10, bbox_to_anchor=(1.3, 1), loc='upper left')
    
    plt.tight_layout()
    
    plt.savefig(r'C:\Users\Fiasco\Desktop\ResBox_new\docs\figs\raw\radar_chart2.tif', transparent=True, dpi=300, bbox_inches='tight')
    plt.show()
    print("\nMean Performance Metrics by Model (Original Scale):")
    print("="*60)
    print(means_df[metrics_to_plot].round(4).to_string())


    model_stats = {}

    for model_name, df in final_analysis.items():
        row = {}
        for m in metrics:
            med = df[m].median()
            q1 = df[m].quantile(0.25)
            q3 = df[m].quantile(0.75)

            row[m] = f"{med:.4g} [{q1:.4g}, {q3:.4g}]"
        model_stats[model_name] = row

    stats_df = pd.DataFrame(model_stats).T

    # stats_df.to_excel(rf"C:\Users\Fiasco\Desktop\ResBox_new\docs\mean_spided_df.xlsx" )


In [ ]:
get_spider(final_analysis)

## sensetivity to specificity

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Custom color palette
colors = colors_base

# Extract the data
specificity_df = result_dict["Specificity"]
sensitivity_df = result_dict["Sensitivity"]
models = specificity_df.columns[1:].tolist()

# Create subplots in a 3x3 grid for better symmetry
fig, axes = plt.subplots(3, 3, figsize=(13, 12))
axes = axes.flatten()

for idx, model in enumerate(models):
    ax = axes[idx]
    
    # Get color for this model
    color = colors.get(model, '#808080')

    spec_values = specificity_df[model].values
    sens_values = sensitivity_df[model].values

    # Plot points
    ax.scatter(spec_values, sens_values, alpha=0.7, s=120,
              edgecolors='black', linewidth=0.5, color=color)

    # Add feature labels
    for i, feature in enumerate(specificity_df['features']):
        # Create shorter, readable labels
        short_feature = feature.replace('com_', '').replace('_', ' ')
        # Truncate if too long
        if len(short_feature) > 25:
            short_feature = short_feature[:25] + '...'

        ax.annotate(f"{i+1}",  # Changed from {i} to {i+1}
                   (spec_values[i], sens_values[i]),
                   fontsize=9,
                   ha='center',
                   va='center',
                   color='white',
                   weight='bold',
                   bbox=dict(boxstyle='circle,pad=0.3',
                           facecolor=color,
                           edgecolor='black',
                           alpha=0.8))

    # Add diagonal reference line
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.3, linewidth=1.5)

    # Add optimal region
    ax.axvline(x=0.8, color='green', linestyle=':', alpha=0.3, linewidth=1.5)
    ax.axhline(y=0.8, color='green', linestyle=':', alpha=0.3, linewidth=1.5)

    ax.set_xlabel('Specificity', fontsize=12, fontweight='bold')
    ax.set_ylabel('Sensitivity', fontsize=12, fontweight='bold')
    ax.set_title(model, fontsize=13, fontweight='bold', pad=10)
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(-0.05, 1.05)

for i in range(len(models), 9):
    axes[i].axis('off')

# plt.suptitle('Specificity vs Sensitivity by Model',
#             fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
# plt.savefig(r'C:\Users\Fiasco\Desktop\ResBox_new\docs\figs\raw\sensitivity_specificity.tif', dpi=300, bbox_inches='tight')
plt.show()

# Print feature legend
print("\n" + "="*90)
print("FEATURE INDEX LEGEND")
print("="*90)
for i, feature in enumerate(specificity_df['features']):
    clean_name = feature.replace('com_', '').replace('_', ' ').title()
    print(f"  {i+1:2d} | {clean_name}")  # Changed from {i:2d} to {i+1:2d}
print("="*90)

## MCC scatter plot

In [ ]:
result_dict.keys()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Define colors for each model
colors = colors_base

# Prepare data
df_mcc = result_dict["MCC"]
x_positions = np.arange(len(df_mcc['features']))

# Create figure
plt.figure(figsize=(13, 6))

# Plot MCC values
for model, color in colors.items():
    plt.scatter(x_positions, df_mcc[model], label=model, color=color, s=100, alpha=0.7)

# Formatting
plt.xlabel('Features', fontsize=12, fontweight='bold')
plt.ylabel('MCC', fontsize=12, fontweight='bold')
plt.xticks(x_positions, df_mcc['features'], rotation=45, ha='right')
plt.ylim(-1, 1)
plt.grid(True, alpha=0.3, linestyle='--')
plt.axhline(y=0, color='black', linestyle='-', linewidth=0.5)

# Legend
plt.legend(fontsize=10, bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
# plt.savefig(r'C:\Users\Fiasco\Desktop\ResBox_new\docs\figs\raw\MCC_scatter.tif', dpi=300, bbox_inches='tight')

plt.show()


## Ratio (Prediction ratio or selection ratio)

In [ ]:
df_base = df_base.drop(columns=["id"]).reset_index(drop=True)

In [ ]:
ratios = {}
for df_name , df_v in model_dfs.items():
    mapper_dic = {}
    for col_name in df_base.columns:
        model_len = len(df_v[df_v[col_name] == True])
        base_len = len(df_base[df_base[col_name] == True])
        mapper_dic[col_name] = model_len/base_len
    ratios[df_name] = mapper_dic

df_ratio = pd.DataFrame(ratios)
df_ratio = df_ratio.rename(index=reverse_mapping)



In [ ]:
# df_ratio.to_excel(r"C:\Users\Fiasco\Desktop\ResBox_new\docs\metrics_ratio.xlsx" )

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

df = df_ratio

# Set up the plot
plt.figure(figsize=(13, 6))

colors = colors_base
x_positions = np.arange(len(df))

# Plot each model
for model in colors.keys():
    plt.scatter(x_positions, df[model], label=model, color=colors[model], s=100, alpha=0.7)

# Customize the plot
plt.xlabel('Features', fontsize=12, fontweight='bold')
plt.ylabel('Prediction ratio', fontsize=12, fontweight='bold')
plt.xticks(x_positions, df.index, rotation=45, ha='right')
plt.yscale('log')

plt.ylim(0.1, 10)  
plt.yticks([0.1, 0.2, 0.5, 1, 2, 5, 10], ['0.1', '0.2', '0.5', '1', '2', '5', '10'])

plt.grid(True, alpha=0.3, linestyle='--')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=10)
plt.axhline(y=1, color='red', linestyle='--', linewidth=1, alpha=0.5)  # Reference line at ratio=1
plt.tight_layout()
plt.savefig(r'C:\Users\Fiasco\Desktop\ResBox_new\docs\figs\raw\ratio_scatter.tif', dpi=300, bbox_inches='tight')
plt.show()

## Missing values

In [ ]:
df_missing = result_dict["Missing_count"]
df_missing["features"] = df_missing["features"].replace(reverse_mapping)


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

df = df_missing

# Set up the plot
plt.figure(figsize=(13, 6))

colors = colors_base

x_positions = np.arange(len(df))
bar_width = 0.12 
offset = np.arange(len(colors)) * bar_width - (len(colors) - 1) * bar_width / 2

for i, model in enumerate(colors.keys()):
    plt.bar(x_positions + offset[i], df[model], width=bar_width, 
            label=model, color=colors[model], alpha=0.7)

plt.xlabel('Features', fontsize=12, fontweight='bold')
plt.ylabel('Missing values', fontsize=12, fontweight='bold')
plt.xticks(x_positions, df['features'], rotation=45, ha='right')
plt.grid(True, alpha=0.3, linestyle='--', axis='y')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=10)
plt.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
plt.tight_layout()
plt.savefig(r'C:\Users\Fiasco\Desktop\ResBox_new\docs\figs\raw\missing_bar.tif', dpi=300, bbox_inches='tight')

# Show the plot
plt.show()